# LeetCode #1388: Pizza With 3n Slices

https://leetcode.com/problems/pizza-with-3n-slices/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(\binom{3n}{n})$ | $O(n)$ |
| **Optimal: Circular House Robber DP ★** | $O(n^2)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Try every combination of `n` non-adjacent slices from the 3n circular arrangement. The binomial coefficient $\binom{3n}{n}$ is astronomically large — unusable beyond tiny inputs.

### Optimal: Circular House Robber DP ★
This is equivalent to the House Robber III problem on a circular array, but instead of "as many as possible", we must pick exactly `n` items. Define `dp[i][j]` = maximum sum choosing `j` non-adjacent elements from the first `i` elements. Run twice to handle circularity (once excluding the last element, once excluding the first), then take the maximum `dp[3n-1][n]`.

**Why this is better than Brute Force:** The 2D DP reuses overlapping sub-problems, reducing the exponential search to $O(n^2)$ time and space.

**Constraints:**
* $1 \leq n \leq 500$
* `slices.length == 3 * n`
* $1 \leq$ `slices[i]` $\leq 1000$

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxSizeSlices(int[] slices) {
        int n = slices.Length / 3;
        // Run twice to handle the circular constraint: fix which end to exclude
        return Math.Max(
            Rob(slices, 0, slices.Length - 2, n),
            Rob(slices, 1, slices.Length - 1, n)
        );
    }

    int Rob(int[] slices, int lo, int hi, int k) {
        int len = hi - lo + 1;
        // dp[i][j] = max sum picking j non-adjacent elements from slices[lo..lo+i)
        var dp = new int[len + 1, k + 1];
        for (int i = 1; i <= len; i++) {
            for (int j = 1; j <= Math.Min(i, k); j++) {
                // Either skip slice i, or take slice i (can't take i-1)
                int take = (i >= 2 ? dp[i - 2, j - 1] : 0) + slices[lo + i - 1];
                dp[i, j] = Math.Max(dp[i - 1, j], take);
            }
        }
        return dp[len, k];
    }
}

### Python

In [ ]:
from typing import List

class Solution:
    def max_size_slices(self, slices: List[int]) -> int:
        n = len(slices) // 3

        def rob(lo: int, hi: int) -> int:
            length = hi - lo + 1
            # dp[i][j] = max sum picking j non-adjacent elements from slices[lo..lo+i)
            dp = [[0] * (n + 1) for _ in range(length + 1)]
            for i in range(1, length + 1):
                for j in range(1, min(i, n) + 1):
                    # Either skip slice i, or take slice i (can't take i-1)
                    take = (dp[i - 2][j - 1] if i >= 2 else 0) + slices[lo + i - 1]
                    dp[i][j] = max(dp[i - 1][j], take)
            return dp[length][n]

        # Run twice to handle the circular constraint: fix which end to exclude
        return max(rob(0, len(slices) - 2), rob(1, len(slices) - 1))

### Go

In [ ]:
func maxSizeSlices(slices []int) int {
    n := len(slices) / 3

    rob := func(lo, hi int) int {
        length := hi - lo + 1
        // dp[i][j] = max sum picking j non-adjacent elements from slices[lo..lo+i)
        dp := make([][]int, length+1)
        for i := range dp { dp[i] = make([]int, n+1) }
        for i := 1; i <= length; i++ {
            for j := 1; j <= i && j <= n; j++ {
                // Either skip slice i, or take slice i (can't take i-1)
                take := slices[lo+i-1]
                if i >= 2 { take += dp[i-2][j-1] }
                if dp[i-1][j] > take { dp[i][j] = dp[i-1][j] } else { dp[i][j] = take }
            }
        }
        return dp[length][n]
    }
    // Run twice to handle the circular constraint: fix which end to exclude
    a := rob(0, len(slices)-2)
    b := rob(1, len(slices)-1)
    if a > b { return a }
    return b
}

### Rust

In [ ]:
impl Solution {
    pub fn max_size_slices(slices: Vec<i32>) -> i32 {
        let n = slices.len() / 3;

        let rob = |lo: usize, hi: usize| -> i32 {
            let length = hi - lo + 1;
            // dp[i][j] = max sum picking j non-adjacent elements from slices[lo..lo+i)
            let mut dp = vec![vec![0i32; n + 1]; length + 1];
            for i in 1..=length {
                for j in 1..=i.min(n) {
                    // Either skip slice i, or take slice i (can't take i-1)
                    let take = (if i >= 2 { dp[i - 2][j - 1] } else { 0 }) + slices[lo + i - 1];
                    dp[i][j] = dp[i - 1][j].max(take);
                }
            }
            dp[length][n]
        };

        // Run twice to handle the circular constraint: fix which end to exclude
        rob(0, slices.len() - 2).max(rob(1, slices.len() - 1))
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `slices = [1,2,3,4,5,6]` (n=2)
Choose 2 non-adjacent slices from [1,2,3,4,5,6] circular. Best: 3+6=9 or 4+1=5... optimal is **10** (4+6, indices 3 and 5 — non-adjacent in both circular runs).

### 2. Slightly Complex
**Input:** `slices = [8,9,8,6,1,1]` (n=2)
Run 1 (exclude last): dp over [8,9,8,6,1], pick 2 non-adjacent — best 8+8=16. Run 2 (exclude first): dp over [9,8,6,1,1], pick 2 — best 9+6=15. Answer: **16**.

### 3. Edge Case: Time Factor
**Input:** n=500, `slices` length 1500.
The DP table is $1500 \times 500 = 750{,}000$ cells, filled twice — $1.5 \times 10^6$ operations total. This is the maximum input and the worst case for time.

### 4. Edge Case: Space Factor
**Input:** n=500, `slices` length 1500.
Two DP tables of size $\approx 1500 \times 500$ are allocated (run twice). At 4 bytes per int, that is $\approx 6$ MB — largest space usage for this problem.

### 5. Almost-Impossible but Plausible
**Input:** `slices = [1000,1000,1000]` (n=1)
Only 3 slices, must pick 1. Circular non-adjacency means we can pick any one slice but not two adjacent. With n=1 any single slice is valid; all are 1000. Answer: **1000**.